# Why Agents Need Standardized Protocols

A single agent calling two in-process functions does not need a protocol. But real systems grow: more agents, more tool servers, more teams, more vendors. Without standards, integration cost explodes as **N agents × M services** — each pair needs a custom adapter, custom auth, and custom documentation.

This notebook explains **why** standardized protocols (like MCP and A2A) exist, with measurable ShopCo examples:

- Quantifying the **N×M adapter problem**
- **Schema drift** and discovery failures in ad-hoc integrations
- **Swapability** when backends change
- **Cross-team** and **cross-org** boundaries
- **Security** and audit at protocol edges
- Honest cases where protocols are **overkill**

Everything runs as self-contained plain Python — no external servers or CLI.

In [ ]:
"""
ShopCo Integration Lab — ad-hoc vs standardized protocols.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- Backend services (simulate separate teams/repos) ---
ORDERS_DB: dict[str, dict[str, Any]] = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICY_DB = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
]

BILLING_DB = {
    "alice@shop.com": {"balance": 0.00, "plan": "pro"},
    "bob@shop.com": {"balance": 12.50, "plan": "basic"},
}

print("ShopCo services: orders, policies, billing")

---

## 1. The Growth Curve — When Ad-Hoc Breaks

| Stage | Agents | Services | Adapters needed (N×M) |
|-------|--------|----------|------------------------|
| Prototype | 1 | 2 | 2 |
| Team scale | 3 | 4 | **12** |
| Platform | 8 | 10 | **80** |

Each adapter is custom code: request shape, auth, error handling, parsing, tests, and docs. Protocols aim for **N + M** — each agent speaks the protocol; each service exposes the protocol.

In [ ]:
def adapter_count(agents: int, services: int, protocol: bool = False) -> int:
    return agents + services if protocol else agents * services


for agents, services in [(1, 2), (3, 4), (8, 10)]:
    ad_hoc = adapter_count(agents, services, protocol=False)
    std = adapter_count(agents, services, protocol=True)
    print(f"{agents} agents × {services} services → ad-hoc: {ad_hoc:>3} adapters | protocol: {std:>3} endpoints")

---

## 2. Ad-Hoc Integration — How It Starts

ShopCo's first support agent calls two services with **bespoke clients** — different URL shapes, field names, and error formats.

In [ ]:
@dataclass
class AdHocAdapter:
    agent_id: str
    service_id: str
    request_shape: dict[str, str]
    loc: int  # lines of code (estimated maintenance burden)


class OrdersServiceAdHoc:
    """Team A's API — uses 'orderId' camelCase."""
    def fetch(self, order_id: str) -> dict:
        oid = order_id.upper()
        row = ORDERS_DB.get(oid)
        if not row:
            return {"success": False, "errorCode": "NOT_FOUND"}
        return {"success": True, "orderId": oid, "orderStatus": row["status"], "total": row["amount"]}


class PolicyServiceAdHoc:
    """Team B's API — uses snake_case and nested 'data'."""
    def search(self, q: str) -> dict:
        terms = [t for t in re.split(r"\W+", q.lower()) if len(t) > 2]
        hits = [p for p in POLICY_DB if any(t in p["text"].lower() for t in terms)] or POLICY_DB[:1]
        return {"data": {"policy_hits": hits}}


class SupportAgentAdHoc:
    def __init__(self):
        self._orders = OrdersServiceAdHoc()
        self._policy = PolicyServiceAdHoc()

    def answer(self, query: str) -> str:
        parts = []
        if "ord-" in query.lower():
            match = re.search(r"ORD-\d{4}", query.upper())
            oid = match.group(0) if match else "ORD-1001"
            raw = self._orders.fetch(oid)
            if raw["success"]:
                parts.append(f"Order {raw['orderId']} is {raw['orderStatus']}")
        if "return" in query.lower() or "policy" in query.lower():
            raw = self._policy.search(query)
            hit = raw["data"]["policy_hits"][0]
            parts.append(f"[{hit['id']}] {hit['text']}")
        return " | ".join(parts) or "No answer"


agent_v1 = SupportAgentAdHoc()
print(agent_v1.answer("Where is ORD-1001 and can I return it?"))

---

## 3. Adding Agents and Services — Adapter Explosion

A billing agent and inventory service arrive. Every agent needs new glue code for every service.

In [ ]:
class BillingServiceAdHoc:
    def lookup(self, email: str) -> dict:
        acct = BILLING_DB.get(email.lower())
        if not acct:
            return {"status": "error", "msg": "no account"}
        return {"status": "ok", "account": {"email": email, **acct}}


class BillingAgentAdHoc:
    def __init__(self):
        self._billing = BillingServiceAdHoc()
        self._orders = OrdersServiceAdHoc()  # duplicate coupling

    def answer(self, email: str) -> str:
        acct = self._billing.lookup(email)
        if acct["status"] != "ok":
            return "Account not found"
        return f"Balance ${acct['account']['balance']:.2f} on {acct['account']['plan']} plan"


AD_HOC_ADAPTERS: list[AdHocAdapter] = [
    AdHocAdapter("support_agent", "orders_service", {"orderId": "string"}, loc=45),
    AdHocAdapter("support_agent", "policy_service", {"data.policy_hits": "array"}, loc=38),
    AdHocAdapter("billing_agent", "billing_service", {"account.email": "string"}, loc=40),
    AdHocAdapter("billing_agent", "orders_service", {"orderId": "string"}, loc=45),  # duplicated!
]

total_loc = sum(a.loc for a in AD_HOC_ADAPTERS)
print(f"Adapters: {len(AD_HOC_ADAPTERS)} | Estimated glue code: {total_loc} LOC")
for a in AD_HOC_ADAPTERS:
    print(f"  {a.agent_id} → {a.service_id} ({a.loc} LOC)")

---

## 4. Failure Mode — Schema Drift

Team A renames `orderStatus` → `fulfillment_state`. Every ad-hoc adapter breaks. Protocols version schemas in **discoverable manifests**.

In [ ]:
class OrdersServiceV2:
    """Breaking change: field renamed."""
    def fetch(self, order_id: str) -> dict:
        oid = order_id.upper()
        row = ORDERS_DB.get(oid)
        if not row:
            return {"success": False}
        return {"success": True, "orderId": oid, "fulfillment_state": row["status"], "total": row["amount"]}


def ad_hoc_call_v2(query: str) -> str:
    svc = OrdersServiceV2()
    raw = svc.fetch("ORD-1001")
    try:
        return f"Order is {raw['orderStatus']}"  # KeyError after rename
    except KeyError as exc:
        return f"BROKEN: {exc} — adapter not updated for v2 API"


print("After Team A ships v2:", ad_hoc_call_v2("ORD-1001"))


# Protocol approach: server publishes schema version in discovery
PROTOCOL_MANIFEST_V2 = {
    "server": "orders",
    "version": "2.0",
    "tools": [{
        "name": "lookup_order",
        "output_fields": ["order_id", "fulfillment_state", "total"],
    }],
}
print("\nProtocol client discovers v2 fields:", PROTOCOL_MANIFEST_V2["tools"][0]["output_fields"])

---

## 5. Failure Mode — Discovery and Documentation Lag

Ad-hoc: developers read wiki pages. Protocol: agents call `tools/list` and get machine-readable schemas.

In [ ]:
WIKI_DOCS = {
    "orders_api": "POST /v2/orders?id=... returns orderStatus (deprecated, use fulfillment_state)",
    "policy_api": "GET /policies?q=... returns data.policy_hits",
}


def ad_hoc_discover_capabilities() -> list[str]:
    """Human reads wiki — error-prone, stale."""
    return ["orders_api (wiki)", "policy_api (wiki)"]


@dataclass
class ProtocolTool:
    name: str
    description: str
    input_schema: dict[str, Any]


PROTOCOL_REGISTRY: list[ProtocolTool] = [
    ProtocolTool("lookup_order", "Fetch order by ORD-####", {"type": "object", "properties": {"order_id": {"type": "string"}}}),
    ProtocolTool("search_policy", "Search policies", {"type": "object", "properties": {"query": {"type": "string"}}}),
    ProtocolTool("lookup_billing", "Billing account by email", {"type": "object", "properties": {"email": {"type": "string"}}}),
]


def protocol_discover() -> list[dict[str, Any]]:
    return [{"name": t.name, "description": t.description, "inputSchema": t.input_schema} for t in PROTOCOL_REGISTRY]


print("Ad-hoc discovery:", ad_hoc_discover_capabilities())
print("Protocol discovery:", [t["name"] for t in protocol_discover()])
print("\nNew agent onboarding: protocol client calls discover once — no wiki hunt.")

---

## 6. Standardized Protocol Layer — ShopCo Protocol Hub

One protocol per service. Agents use a **single client**; services expose a **single server interface**.

In [ ]:
class MessageKind(str, Enum):
    REQUEST = "request"
    RESPONSE = "response"
    ERROR = "error"


@dataclass
class ProtocolMessage:
    id: str
    method: str
    kind: MessageKind
    params: dict[str, Any] = field(default_factory=dict)
    result: Any = None
    error: str | None = None


SERVICE_HANDLERS: dict[str, dict[str, Callable[..., Any]]] = {
    "orders": {
        "lookup_order": lambda order_id: {
            "order_id": order_id.upper(),
            **(ORDERS_DB.get(order_id.upper()) or {"error": "not_found"}),
        },
    },
    "policies": {
        "search_policy": lambda query: [
            p for p in POLICY_DB
            if any(t in p["text"].lower() for t in query.lower().split())
        ] or POLICY_DB[:1],
    },
    "billing": {
        "lookup_billing": lambda email: BILLING_DB.get(email.lower(), {"error": "not_found"}),
    },
}


class ProtocolServer:
    def __init__(self, service_id: str, tools: list[ProtocolTool]):
        self.service_id = service_id
        self.tools = tools
        self._handlers = SERVICE_HANDLERS[service_id]

    def handle(self, msg: ProtocolMessage) -> ProtocolMessage:
        if msg.method == "tools/list":
            return ProtocolMessage(msg.id, msg.method, MessageKind.RESPONSE, result=[
                {"name": t.name, "description": t.description, "inputSchema": t.input_schema} for t in self.tools
            ])
        if msg.method == "tools/call":
            name = msg.params.get("name", "")
            args = msg.params.get("arguments", {})
            fn = self._handlers.get(name)
            if fn is None:
                return ProtocolMessage(msg.id, msg.method, MessageKind.ERROR, error=f"unknown tool {name}")
            return ProtocolMessage(msg.id, msg.method, MessageKind.RESPONSE, result=fn(**args))
        return ProtocolMessage(msg.id, msg.method, MessageKind.ERROR, error=f"unknown method {msg.method}")


class ProtocolClient:
    def __init__(self, server: ProtocolServer):
        self._server = server

    def _call(self, method: str, params: dict | None = None) -> ProtocolMessage:
        req = ProtocolMessage(id=str(uuid.uuid4())[:8], method=method, kind=MessageKind.REQUEST, params=params or {})
        return self._server.handle(req)

    def list_tools(self) -> list[dict]:
        return self._call("tools/list").result

    def call_tool(self, name: str, arguments: dict) -> Any:
        resp = self._call("tools/call", {"name": name, "arguments": arguments})
        if resp.kind == MessageKind.ERROR:
            raise RuntimeError(resp.error)
        return resp.result


SERVERS = {
    "orders": ProtocolServer("orders", [PROTOCOL_REGISTRY[0]]),
    "policies": ProtocolServer("policies", [PROTOCOL_REGISTRY[1]]),
    "billing": ProtocolServer("billing", [PROTOCOL_REGISTRY[2]]),
}

print("Protocol servers:", list(SERVERS.keys()))

---

## 7. One Agent Client — All Services

Any agent connects to any protocol server the same way. No per-service custom glue.

In [ ]:
class UniversalAgent:
    """Single integration pattern for all protocol servers."""

    def __init__(self, agent_id: str, clients: dict[str, ProtocolClient]):
        self.agent_id = agent_id
        self.clients = clients

    def invoke(self, service: str, tool: str, args: dict) -> Any:
        return self.clients[service].call_tool(tool, args)

    def support_answer(self, query: str) -> str:
        parts = []
        if "ord-" in query.lower():
            match = re.search(r"ORD-\d{4}", query.upper())
            if match:
                data = self.invoke("orders", "lookup_order", {"order_id": match.group(0)})
                if "error" not in data:
                    parts.append(f"Order {data.get('order_id', match.group(0))} is {data.get('status')}")
        if "return" in query.lower() or "policy" in query.lower():
            hits = self.invoke("policies", "search_policy", {"query": query[:40]})
            parts.append(f"[{hits[0]['id']}] {hits[0]['text']}")
        return " | ".join(parts) or "How can I help?"

    def billing_answer(self, email: str) -> str:
        data = self.invoke("billing", "lookup_billing", {"email": email})
        if "error" in data:
            return "Account not found"
        return f"Balance ${data['balance']:.2f} ({data['plan']} plan)"


clients = {sid: ProtocolClient(srv) for sid, srv in SERVERS.items()}
support = UniversalAgent("support_agent", clients)
billing = UniversalAgent("billing_agent", clients)

print(support.support_answer("ORD-1001 return policy?"))
print(billing.billing_answer("alice@shop.com"))

---

## 8. Swapability — Change Backend Without Changing Agents

Replace the orders server implementation; agents still call `lookup_order` by name.

In [ ]:
ORDERS_DB_V2 = {k: {**v, "status": v["status"].upper()} for k, v in ORDERS_DB.items()}

SERVICE_HANDLERS["orders"]["lookup_order"] = lambda order_id: {
    "order_id": order_id.upper(),
    "fulfillment_state": ORDERS_DB_V2.get(order_id.upper(), {}).get("status", "UNKNOWN"),
    "source": "v2_warehouse_api",
}

# Agent code unchanged — still calls lookup_order
print("After backend swap (agent unchanged):")
print(support.invoke("orders", "lookup_order", {"order_id": "ORD-1001"}))

---

## 9. Cross-Team Boundaries

Protocols let teams **own** their server. Support team consumes billing tools without importing billing code.

In [ ]:
@dataclass
class TeamOwnership:
    team: str
    service_id: str
    protocol_endpoint: str


TEAM_MAP = [
    TeamOwnership("Orders Squad", "orders", "mcp://orders.shopco.internal"),
    TeamOwnership("Policy Squad", "policies", "mcp://policies.shopco.internal"),
    TeamOwnership("Billing Squad", "billing", "mcp://billing.shopco.internal"),
]


def cross_team_flow(user_query: str) -> dict[str, Any]:
    """Support agent never imports billing module — protocol only."""
    agent = UniversalAgent("support", {sid: ProtocolClient(srv) for sid, srv in SERVERS.items()})
    trace = []
    if "charge" in user_query.lower() or "bill" in user_query.lower():
        data = agent.invoke("billing", "lookup_billing", {"email": "alice@shop.com"})
        trace.append("billing Squad → lookup_billing (protocol)")
        answer = f"Billing: balance ${data.get('balance', 0):.2f}"
    else:
        answer = agent.support_answer(user_query)
        trace.append("orders/policy Squads via protocol")
    return {"answer": answer, "trace": trace, "teams": [t.team for t in TEAM_MAP]}


out = cross_team_flow("Why was I charged on my bill?")
print(out["answer"])
print("Teams involved:", out["trace"])

---

## 10. Cross-Organizational Agents (A2A)

Protocols matter most at **trust boundaries** — when the other agent is outside your company.

In [ ]:
@dataclass
class ExternalAgentCard:
    org: str
    agent_id: str
    skills: list[str]
    auth: str


EXTERNAL_AGENTS = [
    ExternalAgentCard("ShopCo", "shopco-support", ["orders", "policy"], "oauth2"),
    ExternalAgentCard("ShipFast Logistics", "shipfast-tracker", ["shipment_tracking"], "mTLS"),
    ExternalAgentCard("PayRight Inc", "payright-disputes", ["payment_disputes"], "signed_jwt"),
]


def delegate_cross_org(task: str, skill: str) -> dict[str, Any]:
    candidates = [a for a in EXTERNAL_AGENTS if skill in a.skills or any(skill in s for s in a.skills)]
    if not candidates:
        return {"error": "no external agent for skill", "skill": skill}
    agent = candidates[0]
    return {
        "delegated_to": f"{agent.org}/{agent.agent_id}",
        "auth": agent.auth,
        "task": task,
        "result": f"[{agent.org}] handled: {task[:50]}...",
    }


print(delegate_cross_org("Track shipment for ORD-1001", "shipment_tracking"))
print(delegate_cross_org("Dispute duplicate charge", "payment_disputes"))

---

## 11. Security and Audit at the Protocol Edge

Protocols create a clear **audit boundary** — every tool call is a logged message with scope.

In [ ]:
AUDIT_LOG: list[dict[str, Any]] = []


@dataclass
class ScopedConnection:
    agent_id: str
    service_id: str
    allowed_tools: list[str]


def audited_call(scope: ScopedConnection, client: ProtocolClient, tool: str, args: dict) -> Any:
    if tool not in scope.allowed_tools:
        AUDIT_LOG.append({
            "ts": datetime.now(timezone.utc).strftime("%H:%M:%S"),
            "agent": scope.agent_id,
            "service": scope.service_id,
            "tool": tool,
            "allowed": False,
        })
        raise PermissionError(f"{scope.agent_id} cannot call {tool} on {scope.service_id}")
    result = client.call_tool(tool, args)
    AUDIT_LOG.append({
        "ts": datetime.now(timezone.utc).strftime("%H:%M:%S"),
        "agent": scope.agent_id,
        "service": scope.service_id,
        "tool": tool,
        "allowed": True,
        "args_keys": list(args.keys()),
    })
    return result


scope = ScopedConnection("support_agent", "orders", ["lookup_order"])
orders_client = ProtocolClient(SERVERS["orders"])

print(audited_call(scope, orders_client, "lookup_order", {"order_id": "ORD-1001"}))
try:
    audited_call(scope, orders_client, "delete_order", {"order_id": "ORD-1001"})
except PermissionError as e:
    print(f"Blocked: {e}")

print("\nAudit log:")
for entry in AUDIT_LOG:
    print(f"  {entry}")

---

## 12. Integration Cost Model

Compare estimated maintenance burden: ad-hoc vs protocol.

In [ ]:
@dataclass
class IntegrationCostModel:
    agents: int
    services: int
    loc_per_ad_hoc_adapter: int = 42
    loc_per_protocol_endpoint: int = 25
    loc_per_universal_client: int = 30

    def ad_hoc_loc(self) -> int:
        return self.agents * self.services * self.loc_per_ad_hoc_adapter

    def protocol_loc(self) -> int:
        return (
            self.agents * self.loc_per_universal_client
            + self.services * self.loc_per_protocol_endpoint
        )

    def savings_pct(self) -> float:
        ad = self.ad_hoc_loc()
        return 100 * (1 - self.protocol_loc() / ad) if ad else 0


print(f"{'Agents':<8} {'Services':<10} {'Ad-hoc LOC':<12} {'Protocol LOC':<14} {'Savings':<8}")
print("-" * 55)
for agents, services in [(2, 3), (5, 6), (10, 12)]:
    m = IntegrationCostModel(agents, services)
    print(f"{agents:<8} {services:<10} {m.ad_hoc_loc():<12} {m.protocol_loc():<14} {m.savings_pct():.0f}%")

---

## 13. Side-by-Side — Ad-Hoc vs Protocol

| Dimension | Ad-hoc | Standardized protocol |
|-----------|--------|----------------------|
| Integration count | N × M | N + M |
| Discovery | Wiki / Slack | `tools/list`, Agent Card |
| Schema changes | Break all adapters | Versioned manifest |
| New agent onboarding | Write M adapters | Connect M servers (same client) |
| Cross-team | Import their code | Connect to their server |
| Cross-org | Custom legal + tech | A2A + standard auth |
| Audit | Scattered logs | Protocol message log |
| Swap backend | Rewrite adapters | Swap server, same tool names |

---

## 14. When Protocols Are Overkill

Honest tradeoffs — do not add protocol overhead everywhere.

| Skip protocols when… | Why |
|---------------------|-----|
| 1 agent, 2 in-process tools | Direct function calls are simpler |
| Prototype with 1-week lifetime | Integration cost never amortizes |
| Single team owns everything | No boundary to standardize |
| Ultra-low latency inside one process | Protocol adds serialization hop |
| Tools never change | Discovery value is low |

**Rule of thumb:** adopt protocols when you have **multiple agents** or **multiple owning teams**, or when tools live **outside your process**.

In [ ]:
def recommend_protocol(agents: int, services: int, cross_team: bool, external_agents: bool) -> str:
    if agents <= 1 and services <= 2 and not cross_team and not external_agents:
        return "skip — use in-process tools"
    if external_agents:
        return "yes — A2A for external agents, MCP for tools"
    if cross_team or agents * services > 6:
        return "yes — MCP-style tool protocols"
    return "maybe — monitor adapter growth"


SCENARIOS = [
    (1, 2, False, False, "Solo prototype"),
    (3, 4, True, False, "Multi-team platform"),
    (2, 3, False, True, "Partner logistics agent"),
]

for agents, services, cross, external, label in SCENARIOS:
    rec = recommend_protocol(agents, services, cross, external)
    print(f"{label:<28} → {rec}")

---

## 15. The Five Core Reasons — Summary Table

| # | Reason | ShopCo demo |
|---|--------|-------------|
| 1 | **Scale** — avoid N×M adapters | 8×10 = 80 → 18 endpoints |
| 2 | **Discovery** — machine-readable capabilities | `tools/list` vs wiki |
| 3 | **Evolution** — schema drift without breakage | `fulfillment_state` rename |
| 4 | **Boundaries** — teams and orgs | Billing Squad owns server |
| 5 | **Governance** — audit and scope | `ScopedConnection` + log |

---

## 16. Check Your Understanding

1. Why does integration cost grow as **N×M** without protocols?
2. How does **schema drift** break ad-hoc adapters?
3. What does **discovery** give you that a wiki does not?
4. Why can agents stay unchanged when a backend is swapped under a protocol?
5. When is **cross-organizational** delegation impossible without a standard like A2A?
6. What is audited at the **protocol edge**?
7. When should you **skip** protocols and use in-process tools?

---

## 17. Summary

Agents need standardized protocols when systems **scale beyond one team and one process**:

- **N×M → N+M** — slash adapter count and maintenance LOC.
- **Discovery** — agents find tools programmatically, not from stale docs.
- **Evolution** — servers version capabilities; clients discover changes.
- **Boundaries** — teams and vendors expose servers, not shared code imports.
- **Governance** — scoped connections and message-level audit trails.

ShopCo's ad-hoc path broke on schema rename and adapter duplication. The protocol hub let support and billing agents share one client pattern across orders, policy, and billing services. Use protocols when boundaries multiply — skip them when a simple in-process call still wins.